# Exploring Microsoft Qlib on Google Colab

This notebook is a **guided tour of Microsoft [Qlib](https://github.com/microsoft/qlib)**. It installs the library, downloads a small dataset, runs a baseline forecasting experiment, and points you to the most important source files to read inside the cloned repository.

> Run the cells from top to bottom in Google Colab. Sections marked *optional* can be skipped if you only want to read the explanations.

## 0. Prerequisites and runtime notes
- The notebook works in a free Colab runtime (CPU is fine; GPU is optional for deep-learning benchmarks).
- Expected runtime: 5–10 minutes for installation and data download, plus ~2–3 minutes for the baseline training run.
- All paths assume the Colab working directory (`/content`).

In [ ]:
# Check environment details (helpful for troubleshooting)
import platform, sys, subprocess
print("Python:", sys.version)
print("Platform:", platform.platform())
print("GPU available:" , subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader'))

## 1. Install Qlib and clone the repository
These commands install the published package, clone the GitHub repository for reference, and install example dependencies.

- `pip install qlib`: core library from PyPI.
- `git clone https://github.com/microsoft/qlib`: fetch the source for browsing.
- `pip install -r requirements.txt`: extra packages used by examples (some may be optional; you can re-run without this if any dependency fails to build).

In [ ]:
!pip -q install qlib
!git clone --depth 1 https://github.com/microsoft/qlib.git
%cd qlib
!pip -q install -r requirements.txt
%cd /content

## 2. Download sample market data (Yahoo CN) *optional but recommended*
Qlib needs a local data directory. The helper script below downloads a small **daily** dataset for the China market (CN). You can switch to US data by setting `--region us`.

- Target directory: `/content/qlib_data/cn_data`
- Script: `scripts/get_data.py`
- Expected download time: ~2–4 minutes depending on network speed.

In [ ]:
%cd /content/qlib
!python scripts/get_data.py --target_dir /content/qlib_data/cn_data --region cn
%cd /content

## 3. Repository tour
The snippet below prints a concise directory overview so you can map documentation to code:

- `qlib/`: core package (data providers, models, workflow utilities).
- `examples/`: runnable training and analysis scripts.
- `scripts/`: data preparation helpers (Yahoo, CSI, Alpha158 features, etc.).
- `benchmarks/`: configuration presets for common models.

In [ ]:
import os, textwrap
root = '/content/qlib'
keys = ['qlib', 'examples', 'scripts', 'benchmarks', 'docs']
rows = []
for k in keys:
    path = os.path.join(root, k)
    if not os.path.isdir(path):
        continue
    children = [c for c in sorted(os.listdir(path)) if not c.startswith('.')][:12]
    rows.append(f"{k}/ -> {', '.join(children)}")
print("
".join(rows))

## 4. Initialize Qlib and load sample features
This cell initializes Qlib, sets the data directory, and fetches a small feature DataFrame so you can confirm data access.

Key APIs:
- `qlib.init(provider_uri=..., region=REG_CN)`: points to the downloaded CN data.
- `D.list_instruments`: lists tickers in a market and date range.
- `D.features`: pulls arbitrary columns (e.g., `$close`, `$volume`) for a given universe and date range.

In [ ]:
from qlib.data import D
import qlib
from qlib.config import REG_CN

# Initialize using the downloaded CN data
qlib.init(provider_uri='/content/qlib_data/cn_data', region=REG_CN)

# Query a small universe and inspect basic features
market = 'csi300'
universe = D.list_instruments(market=market, start_time='2022-01-01', end_time='2022-03-01')[:5]
features = D.features(universe, ['$close', '$volume'], start_time='2022-01-01', end_time='2022-03-01')
features.head()

## 5. Run a baseline forecasting experiment
The `examples/train.py` script supports configuration-driven experiments. Here we run a **single-factor linear regression** benchmark on the CN daily dataset.

- Config file: `examples/benchmarks/Linear/train_cn.yaml`
- Data directory: `/content/qlib_data/cn_data`
- Artifacts: metrics and predictions are saved under `examples/mlruns` (MLflow) and `examples/output`.

In [ ]:
%cd /content/qlib/examples
!python train.py --config benchmarks/Linear/train_cn.yaml --task benchmark --data_dir /content/qlib_data/cn_data --experiment_name demo_linear
%cd /content

## 6. Inspect metrics and artifacts
Use the MLflow run directory to find logs, metrics, and model checkpoints. The snippet below locates the latest run and previews any generated prediction files.

In [ ]:
import pathlib, pandas as pd
run_root = pathlib.Path('/content/qlib/examples/mlruns')
if run_root.exists():
    latest = max(run_root.rglob('meta.yaml'), key=lambda p: p.stat().st_mtime).parent
    print('Latest MLflow run directory:', latest)
else:
    print('No MLflow runs found yet. Did the training cell complete successfully?')

# Preview predictions if available
pred_files = sorted(pathlib.Path('/content/qlib/examples/output').glob('**/pred.pkl'))
if pred_files:
    df = pd.read_pickle(pred_files[-1])
    display(df.head())
else:
    print('No prediction files found. Check the training logs above for errors.')

## 7. Where to read the code (quick map)
Focus on these modules when browsing the cloned repository:

- **Data layer** (`qlib/data`): providers (`provider.py`), dataset handlers (`dataset/handler.py`), and feature engineering (`dataset/feature.py`).
- **Model implementations** (`qlib/model`): base classes (`base.py`), classic ML models (`model.py`), and PyTorch utilities (`pytorch_graph.py`).
- **Workflow & experimentation** (`qlib/workflow`): experiment manager (`task/train.py`), collectors (`task/collector.py`), and MLflow integration (`logger.py`).
- **Examples & benchmarks** (`examples/benchmarks`): YAML configs for linear, MLP, GRU, and transformer models; inspect `train_cn.yaml` for hyperparameters.
- **Data preparation scripts** (`scripts/data_collector`): Yahoo/CSI downloaders (`collector.py`), Alpha158 feature generator (`alpha158.py`).

Tip: use `!sed -n '1,120p qlib/qlib/data/provider.py'` in a notebook cell to read source snippets inline.

## 8. Troubleshooting and next steps
- **Installation errors**: rerun the install cell without `requirements.txt` if a heavy optional dependency fails; you can add specific packages later (e.g., `lightgbm`).
- **Data download slow**: switch to US data (`--region us`) or reduce the date span in the feature query.
- **Trying deep-learning models**: point `--config` to `benchmarks/GRU/train_cn.yaml` or `benchmarks/Transformer/train_cn.yaml` and enable a GPU runtime in Colab.
- **Backtesting & portfolio analysis**: explore `examples/port_analysis` and `benchmarks/PortAna` for end-to-end portfolio evaluation.

Happy researching!